# Activity 4: AutoML and the Limits of Automation

**Week 6 Day 2 · Machine Learning for Data Cleaning**

## The pitch, and the question behind it

AutoML tools promise to do the modelling for you. Point one at a table, name the target column, wait a few minutes, and receive a trained, tuned, ensembled model with a full evaluation report.

That promise is largely real. The tools genuinely work, and they will often beat a competent engineer's first hand built attempt.

Which raises the question this activity exists to answer:

> **If the tool does the modelling, what is left for you to do?**

The answer turns out to be almost everything that matters, and you are going to prove it experimentally rather than take it on faith.

## Why a data engineer should care about AutoML

You are not being trained as a machine learning engineer. You should still know these tools, for three practical reasons:

1. **Baselines are cheap.** When someone claims a model is needed, AutoML tells you in ten minutes what is achievable. Sometimes the answer is "barely better than guessing", which is extremely useful to know before a project is staffed.
2. **The artifacts are reusable.** Good AutoML tools emit feature importance, correlation analysis, and per model diagnostics. That output is valuable for understanding a dataset even when you never deploy the model.
3. **You will be asked to operate them.** Someone will hand you an AutoML model and ask you to run it nightly. Knowing what it does and does not check is your job.

## Learning objectives

By the end of this activity you will be able to:

1. Run an AutoML experiment end to end and interpret the leaderboard.
2. Compare any model against a trivial baseline before accepting that it is good.
3. Navigate the artifacts a good AutoML tool produces.
4. Detect target leakage that AutoML reported as excellent performance.
5. Choose between AutoML libraries, and explain which are viable on this project's Python version.
6. State precisely which parts of the pipeline AutoML does not automate.

---
## Setup

We use **MLJAR** (`mljar-supervised`). It was chosen for this course because it generates the richest set of explanatory artifacts of any of the open source options, and because it works on this project's Python version, which is not true of every alternative. More on that in Part 6.

`mljar-supervised` and `scikit-learn` are declared in the repo-root `pyproject.toml`, so `uv sync` from the repository root installs them. Run that once if you have not since they were added.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 30)

The next cell builds `DATA_DIR` the same way as Activities 1 through 3: by walking up from wherever the notebook is running until it finds `pyproject.toml`, the marker for the repository root. That means the path resolves correctly whether you run this notebook from the course folder or from your own copy under `student-work/week6/day2/`, without you hardcoding it.

In [ ]:
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find the repo root (the folder holding pyproject.toml)."""
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from " + str(start))


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "Week 6" / "Labs" / "Day 2" / "data"

# AutoML writes a lot of files. Keep them under student-work/ so they never
# collide with the course folder you pull updates into.
OUTPUT_DIR = REPO_ROOT / "student-work" / "week6" / "day2" / "automl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data   :", DATA_DIR)
print("Output :", OUTPUT_DIR)

---
# Part 1: The dataset

The German Credit dataset. Each row is a loan applicant, and the target `credit_risk` records whether the loan turned out good or bad.

In [ ]:
credit = pd.read_csv(DATA_DIR / "credit_data" / "credit_data_train.csv")
print(credit.shape)
credit.head()

In [ ]:
credit["credit_risk"].value_counts()

Note the imbalance: roughly 70 percent good, 30 percent bad. Write that number down, because it is the number every model has to beat.

## The trivial baseline

Before running anything, establish what "doing nothing" scores. A model that always predicts the majority class needs no training and no data.

In [ ]:
majority_class_accuracy = (credit["credit_risk"] == "good").mean()
print(f"Always predict 'good': {majority_class_accuracy:.2%} accuracy")

**70.35 percent, for free.**

Any model that does not clearly beat this has produced nothing of value, regardless of how sophisticated it looks. This is the same discipline from Activity 1, where mean imputation was the number to beat, and Activity 3, where a single engineered feature was the number to beat.

---
# Part 2: Running AutoML the way most people first run it

We are going to do this the naive way first: hand the tool the file as it came, name the target, and let it work.

This is exactly what happens in practice when someone is in a hurry.

## Why we split the data first

AutoML does its own internal validation, so it may look redundant to hold rows back. It is not, and the reason is the whole point of this activity.

The tool uses its internal validation to **choose** a model. Any number it reports afterwards is a score on data it used while deciding, and a score you used for selection is optimistic almost by construction. To find out how the winner performs, you need rows that took no part in any decision.

So we cut 20 percent off now, hide it, and do not touch it until the very end.

`stratify=target` keeps the same 70/30 good-to-bad ratio in both halves. Without it a random split can hand you a holdout set with a noticeably different class balance, and your accuracy would then partly measure that accident rather than the model.

In [ ]:
from sklearn.model_selection import train_test_split

target = credit["credit_risk"]
predictors = credit.drop(columns=["credit_risk"])

X_train, X_test, y_train, y_test = train_test_split(
    predictors, target, test_size=0.2, random_state=42, stratify=target
)
print("Training rows:", len(X_train), " Holdout rows:", len(X_test))

In [ ]:
import shutil

from supervised import AutoML


def run_automl(results_path, X, y, mode="Explain", total_time_limit=60):
    """Train a fresh AutoML run, clearing any previous run at this path first.

    MLJAR will not retrain into a folder that already holds a finished run. It
    reloads the old model instead, which means an edited notebook can silently
    report the previous run's results. Clearing the folder makes every run
    reproducible, which matters more here than saving a few seconds.
    """
    results_path = Path(results_path)
    if results_path.exists():
        print(f"Clearing previous run at {results_path.name}/")
        shutil.rmtree(results_path)

    model = AutoML(
        mode=mode,
        total_time_limit=total_time_limit,
        results_path=str(results_path),
        random_state=42,
        verbose=0,
    )
    model.fit(X, y)
    return model


def leaderboard_of(automl):
    """Leaderboard with metric_value guaranteed numeric and one row per model."""
    board = automl.get_leaderboard().copy()
    board["metric_value"] = pd.to_numeric(board["metric_value"])
    return board.drop_duplicates(subset="name").reset_index(drop=True)

The `run_automl` helper defined above wraps four parameters that are doing the real work.

- **`mode="Explain"`** is the preset. It favours speed and explanatory output over the last fraction of accuracy, and it is what makes the tool write all the charts and reports you will tour in Part 4. Part 5 covers the other modes.
- **`total_time_limit=60`** is a ceiling, not a target. `Explain` mode on a dataset this small usually finishes in **15 to 25 seconds**, so do not worry when it returns well before a minute.
- **`results_path`** is the folder it writes everything to.
- **`verbose=0`** silences the per model progress log.

## Why the helper deletes the folder first

This is worth understanding, because it is a real trap rather than housekeeping.

If `results_path` already contains a finished run, MLJAR does **not** retrain. It reloads the old model from disk instead. So if you change the features and re-run the cell, you can get the **previous** model's results with nothing but a printed notice to tell you.

The reload path is also less reliable than a fresh fit. Depending on the version it can hand back metric values as strings instead of numbers, duplicate rows in the leaderboard, or fail outright when the constructor arguments no longer match what is stored on disk.

None of that is worth working around in a teaching notebook, so `run_automl` clears the folder and always trains from scratch. It costs about twenty seconds and buys you results that always match the code you are looking at.

**The transferable lesson:** when a tool caches expensive work, find out exactly what invalidates the cache before you trust a number that came out of it. "Why did my change have no effect?" is very often this.

In [ ]:
NAIVE_PATH = OUTPUT_DIR / "naive"

naive_automl = run_automl(NAIVE_PATH, X_train, y_train)

That trained several model families, tuned them, and built an ensemble. Look at the leaderboard.

Two things to know before you read it.

**The `metric_value` column is logloss, not accuracy, and lower is better.** Logloss scores the *probability* a model assigned to the correct answer rather than just whether it got the answer right. A model that says "90 percent good" and is right scores well; one that says "51 percent good" and is right scores poorly, because it was barely committed. Getting a confident prediction wrong is punished hardest of all.

AutoML tools optimise logloss rather than accuracy because it responds to small improvements. Accuracy only moves when a prediction crosses the 50 percent line, so it gives the search almost nothing to follow.

**These are internal validation scores, not holdout scores.** They come from data the tool used while choosing, so they are the numbers it used to rank itself. Treat the leaderboard as a comparison *between* models, and never as an estimate of real world performance.

In [ ]:
leaderboard_of(naive_automl)

Now the number that matters: performance on the holdout set the model never saw.

### Predict before you run

Credit risk is hard. The baseline is 70.35 percent. Six model families just competed, and the best internal logloss was only modestly better than the "always guess good" baseline.

**What holdout accuracy do you expect?** Something in the low to mid seventies would be a sensible guess, and it is roughly what the leaderboard implies.

Commit to a number, then run the cell.

In [ ]:
from sklearn.metrics import accuracy_score

naive_predictions = naive_automl.predict(X_test)
naive_accuracy = accuracy_score(y_test, naive_predictions)

print(f"AutoML holdout accuracy : {naive_accuracy:.2%}")
print(f"Trivial baseline        : {majority_class_accuracy:.2%}")
print(f"Improvement             : {naive_accuracy - majority_class_accuracy:+.2%}")

Around **94 percent accuracy**, against a 70 percent baseline. A 24 point improvement in 60 seconds of compute, with no feature engineering and no domain knowledge.

If you predicted the low seventies, your prediction was better calibrated than the result. Hold onto that discomfort, because it is the correct instinct and the rest of the activity is about acting on it.

Now go back and look at the leaderboard you already printed, because **the warning was in it and we walked straight past it**.

`1_Baseline` scored about **0.61** logloss. The Ensemble scored about **0.195**, roughly three times better, and Xgboost and Random Forest both landed near 0.20. On a problem where published results sit in the 70 to 78 percent accuracy range, a model family tripling the baseline's logloss is not a good day at the office. It is a red flag.

That is the habit worth building: **read the leaderboard before you read the accuracy**, and treat a suspiciously large gap over `1_Baseline` as something to explain rather than something to celebrate.

This is the point at which the result gets reported to a stakeholder and the project is declared a success.

Do not report it yet.

---
# Part 3: When a result is too good, find out why

Credit risk is a genuinely hard problem. Banks employ teams of people on it. Published results on this dataset typically land in the 70 to 78 percent range.

We just got 94 percent, from default settings, in one minute.

**A result far better than the field achieves is evidence of a bug, not a breakthrough.** Go and find it.

The tool told us which features it relied on. Read that.

In [ ]:
# Feature importance for the Random Forest, the last model folder alphabetically.
# Every model folder has its own importance file; they broadly agree here.
importance_files = sorted(NAIVE_PATH.glob("*/learner_fold_0_importance.csv"))
print("Models with an importance file:", [f.parent.name for f in importance_files])
print()

importance = pd.read_csv(importance_files[-1])
importance.columns = ["feature", "importance"]
print("Reading:", importance_files[-1].parent.name)
importance.nlargest(10, "importance")

The `id` column dominates.

`id` is a row identifier. It is assigned by whatever system exported this file. It describes nothing about the applicant, their income, their history, or their loan.

It should carry no predictive information whatsoever. Check what it is actually doing.

In [ ]:
credit.groupby("credit_risk")["id"].agg(["min", "max", "mean"]).round(1)

There it is.

Applicants with `good` credit have a mean `id` around 367. Applicants with `bad` credit have a mean around 809. **The file was sorted by outcome before the identifiers were assigned.**

So `id` predicts the target almost perfectly, within this file. The model found that relationship immediately and leaned on it, because it is by far the strongest signal available.

This is **target leakage**: information about the answer leaking into the inputs through a column that will not carry that information in production. When this model meets real applicants, whose IDs are assigned in arrival order, the column it depends on most will be pure noise.

Remove it and measure honestly.

In [ ]:
X_train_clean = X_train.drop(columns=["id"])
X_test_clean = X_test.drop(columns=["id"])

In [ ]:
HONEST_PATH = OUTPUT_DIR / "honest"

honest_automl = run_automl(HONEST_PATH, X_train_clean, y_train)

In [ ]:
honest_accuracy = accuracy_score(y_test, honest_automl.predict(X_test_clean))

results = pd.DataFrame({
    "setup": ["Trivial baseline", "AutoML with leaked id", "AutoML, id removed"],
    "holdout_accuracy": [majority_class_accuracy, naive_accuracy, honest_accuracy],
})
results["vs_baseline"] = results["holdout_accuracy"] - majority_class_accuracy
results.round(4)

In [ ]:
leak_check = (
    leaderboard_of(naive_automl)[["name", "metric_value"]]
    .merge(
        leaderboard_of(honest_automl)[["name", "metric_value"]],
        on="name", suffixes=("_with_id", "_without_id"),
    )
)
leak_check["got_worse_by"] = leak_check["metric_value_without_id"] - leak_check["metric_value_with_id"]
leak_check.round(3)

This is the leakage seen from the model's side rather than the metric's.

`1_Baseline` is unchanged, exactly as it must be: it ignores the features entirely, so removing a column cannot touch it. It is the control in this experiment.

Every model that actually uses features got **substantially worse** once `id` was removed, and the ones that got worst were the ones that had leaned hardest on it. That is what a leaked column looks like when you take it away: performance collapses toward the baseline, because the thing holding the model up was never real.

Run this comparison whenever you suspect leakage. **Drop the suspect column, refit, and see how much you lose.** If a single identifier column is worth a third of your logloss, it was not a feature, it was the answer key.

## Read that table carefully

The real model scores about **73 percent** against a **70 percent** baseline. Roughly **three points** of genuine improvement.

Not 24. Three.

Everything above that was an artifact of how the CSV happened to be sorted.

### What this tells you about AutoML

Consider everything the tool did during the first run. It trained six model families. It tuned them. It built an ensemble. It computed feature importance, generated SHAP explanations, plotted ROC curves, learning curves, and confusion matrices, and wrote a full HTML report.

**At no point did it ask whether `id` should be in the model.**

It cannot. The tool optimises a metric using the columns it is given. Deciding which columns are legitimate requires knowing what the data means, where it came from, and what will exist at prediction time. That is not a modelling question. It is a data engineering question, and no amount of automation touches it.

This is the honest answer to the question at the top of the notebook. AutoML automates model selection and hyperparameter tuning, which are the parts that were already fairly mechanical. What it does not automate:

- Deciding which columns may legitimately be used
- Detecting leakage
- Cleaning missing data correctly, per Activity 1
- Deciding whether outliers are errors or real events, per Activity 3
- Choosing granularity and reference populations, per Activity 2
- Judging whether three points over baseline justifies deploying anything

Every one of those is upstream of the model, and every one belongs to you.

### Try it yourself

You have seen leakage diagnosed after the fact. Now catch it in ten seconds, without training anything.

The `id` column leaked because it was correlated with the target. You do not need an AutoML run to notice that.

1. Convert `credit_risk` to 0 and 1.
2. Compute the correlation between that and every numeric column, including `id`.
3. Sort by absolute correlation.

**What you should see:** `id` at the top by a wide margin, far ahead of any genuine feature such as `duration` or `amount`.

**The point:** a two line correlation check would have caught this before a single model was trained. Cheap screens run first. Expensive tools confirm what the cheap screens found, they do not replace them.

**Then think about the limit of this technique.** Correlation only sees straight-line relationships between numeric columns. Write yourself one sentence on what kind of leakage this screen would miss entirely.

In [ ]:
# Your turn: rank every numeric column by how strongly it correlates with the target.
# Hint: credit.select_dtypes("number").corrwith(target_as_0_1).abs().sort_values(...)



---
# Part 4: The artifacts, which are the real product

Even at three points of lift, the run was worthwhile, because of what it wrote to disk. MLJAR was chosen for this course specifically for this.

In [ ]:
artifacts = sorted(p.name for p in HONEST_PATH.iterdir())
for name in artifacts:
    print(" ", name)

In [ ]:
# Look inside one real model folder. The Ensemble folder is a special case
# with fewer files, so pick a trained model family instead.
example = HONEST_PATH / "6_Default_RandomForest"

print("Artifacts inside", example.name, ":")
for f in sorted(example.iterdir()):
    print("  ", f.name)

For **every model it trained**, you get a confusion matrix, ROC curve, precision recall curve, lift and cumulative gains curves, learning curves, a calibration curve, and permutation importance. Most models also get a SHAP summary plot showing which features pushed individual predictions which way.

Producing that by hand for six model families is most of a day's work. Here it is a side effect.

The top level `README.md` is a full comparison report. Open it in VS Code to read it, and open any individual model folder's `README.md` for that model's detail. The path was printed by the setup cell if you need it.

Two of the top level files are worth opening on any new dataset, regardless of whether you keep the model:

- **`correlation_heatmap.png`** shows which columns move together. Pairs at near perfect correlation usually mean a duplicated or derived column.
- **`features_heatmap.png`** shows which features each model leaned on, so you can see at a glance whether every model agrees about what matters. Unanimous reliance on one column is the fingerprint you went hunting for in Part 3.

In [ ]:
leaderboard = leaderboard_of(honest_automl)
leaderboard

Read the leaderboard as a diagnostic, not just a ranking. Three things in it are worth more than the ordering.

**`1_Baseline` is a real competitor.** MLJAR always trains a "predict the majority class" model first, the same trivial baseline you computed by hand in Part 1, and it scores about **0.61** logloss. That row exists so you can never accidentally celebrate a model that has not beaten doing nothing.

**Two of the six models lose to it.** The Decision Tree comes in around **0.71** and the Neural Network around **0.66**, both *worse* than the baseline. This is normal and worth internalising: a model family being sophisticated says nothing about whether it suits your data, and on a small tabular dataset a neural network is usually the wrong tool.

**The winners are bunched.** Xgboost lands near **0.53**, Random Forest near **0.56**, and the Ensemble at **0.528**, barely ahead of the best single model. When the top of a leaderboard is packed this tightly, the tool is telling you that **the remaining error is in the data, not the algorithm**. Switching models will not fix it. Better features might.

That third reading is the valuable one to deliver in week one of a project, and it takes a minute to produce. "We ran six model families and they all agree the ceiling is here" is a much stronger statement than any single accuracy number.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sorted_lb = leaderboard.sort_values("metric_value")

baseline_score = leaderboard.loc[leaderboard["name"] == "1_Baseline", "metric_value"].iloc[0]
colors = ["tab:red" if v > baseline_score else "tab:blue" for v in sorted_lb["metric_value"]]

ax.barh(sorted_lb["name"], sorted_lb["metric_value"], color=colors)
ax.axvline(baseline_score, color="black", ls="--", lw=1.5,
           label=f"1_Baseline ({baseline_score:.3f})")
ax.set_xlabel("logloss (lower is better)")
ax.set_title("Red bars are models that lost to predicting the majority class")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## A note on this dataset and fairness

This dataset contains `age`, `foreign_worker`, and `people_liable`. It is a credit decisioning dataset, which is a legally regulated application in most jurisdictions.

A model that uses those columns to decide who receives credit can produce discriminatory outcomes even when no one intended it, and even when the protected attribute is not used directly, because other columns act as proxies for it.

MLJAR has built in fairness support through its `fairness_metric` and `privileged_groups` parameters. We are not running a full fairness analysis here, but you should know two things:

1. Accuracy alone is never a sufficient evaluation for a decision that affects people.
2. "The AutoML tool chose the model" is not a defence anyone will accept.

If you work on this kind of system, fairness assessment is a required step, not an optional extra.

---
# Part 5: Modes, and buying accuracy with time

MLJAR exposes presets that trade compute against thoroughness.

| Mode | Purpose | Typical use |
| :--- | :--- | :--- |
| `Explain` | Fast, heavily documented, full explanatory artifacts | Understanding a new dataset. What we used. |
| `Perform` | Balanced, tuned for a deployable model | Building something you intend to ship |
| `Compete` | Exhaustive search, stacking, long runtimes | Squeezing out the last fraction of a point |
| `Optuna` | Optuna driven hyperparameter search | When you have hours of compute available |

The important question is not which is best, but **what the extra compute buys you**.

Given the leaderboard you just saw, where every model clustered near the baseline, spending an hour in `Compete` mode would likely gain a fraction of a percentage point on a model that is three points above guessing.

Knowing when **not** to spend the compute is the skill.

---
# Part 6: Choosing an AutoML library

Several mature options exist. They are not equally usable on any given project, and compatibility is a real constraint rather than a footnote.

| Library | Maintainer | Strengths | Watch out for |
| :--- | :--- | :--- | :--- |
| **MLJAR** (`mljar-supervised`) | MLJAR | Best in class explanatory artifacts, readable reports, built in fairness metrics | Slower than lightweight options on large data |
| **H2O AutoML** | H2O.ai | Distributed, scales to large data, strong stacked ensembles | Requires a running JVM, which complicates container setup |
| **AutoGluon** | AWS | Very strong accuracy, handles text and images as well as tables | Heavy install, large dependency footprint |
| **FLAML** | Microsoft | Fast and low resource, good cost aware search | Far fewer artifacts, minimal reporting |
| **PyCaret** | Community | Friendly API, very popular in tutorials | **Not installable on this project.** See below. |

## A concrete compatibility warning

This project targets Python 3.13 and pandas 3.x. PyCaret 3.3.2, the current release, cannot be installed into it. Its dependency `pmdarima` has no wheels built for Python 3.13 and fails to compile, and forcing a resolution pins the environment back to numpy 1.26 and pandas 2.1, which would downgrade the entire repository.

That is worth internalising as a general lesson. **A library's popularity in tutorials tells you nothing about whether it will install in your environment.** Check compatibility against your actual Python and dependency versions before designing a pipeline around a tool, because discovering this after committing to an architecture is expensive.

If you want PyCaret specifically, it needs its own isolated environment on an older Python. That is a legitimate choice, and it is a cost you should decide on deliberately rather than discover.

---
# Your Turn

Work in your own copy under `student-work/week6/day2/`.

Challenges 1 and 3 are the core. Challenge 2 is a short experiment, and 4 and 5 are the written deliverables.

Use the `run_automl` helper for anything that refits, or pass a new `results_path` yourself. Fitting into a folder that already holds a run will not retrain.

## Challenge 1: Hunt for more leakage

You removed `id`. Confirm nothing else is leaking.

1. For each remaining column, measure how strongly it alone predicts `credit_risk`. A single feature decision tree or a simple group by rate is fine.
2. Identify any column that looks suspiciously predictive.
3. For each suspicious column, decide whether it would genuinely be available **at the moment a credit decision is made**.
4. Report your conclusion, including columns you investigated and cleared.

**What you should see:** nothing remotely like `id`, which outscores every real column by roughly six times. The strongest genuine predictors are `status`, `credit_history`, and `duration`, and all three are legitimately known at application time. Expect to clear everything, and say so explicitly.

**Watch for the trap:** `status` and `credit_history` are the two most predictive remaining columns and neither is leakage, because an applicant's existing accounts and borrowing history are known before the loan is granted. Predictive is not the same as leaking. The test is always availability at prediction time, never strength.

**Success criteria**
- [ ] Every remaining column screened, including the categorical ones.
- [ ] Each suspicious column judged on availability at decision time, with a stated verdict.
- [ ] Columns you checked and cleared are listed, not just the ones you removed.

## Challenge 2: Make the compute pay, or show it does not

1. Re-run AutoML in `Perform` mode with a 300 second limit on the cleaned data, into a new results folder.
2. Compare holdout accuracy against the 60 second `Explain` run.
3. Calculate the accuracy gained per additional minute of compute.
4. State whether you would spend it, and what accuracy gain would change your answer.

**What you should see:** a small gain at best, possibly none, for roughly five times the compute. A worse result than the shorter run is entirely possible on a dataset this size, and is a legitimate finding rather than a mistake.

**Success criteria**
- [ ] Both runs' holdout accuracy reported side by side.
- [ ] Gain expressed per unit of compute, not just as a raw difference.
- [ ] A stated decision plus the threshold that would reverse it.

## Challenge 3: Accuracy is the wrong metric here

The classes are imbalanced, and the two errors have very different costs. Approving a bad loan loses money. Declining a good applicant loses a customer and may raise fairness concerns.

1. Produce a confusion matrix for the cleaned model on the holdout set.
2. Compute precision, recall, and F1 for the `bad` class specifically.
3. Explain why 73 percent accuracy is a misleading headline on this dataset.
4. Recommend the metric you would actually optimise, and justify it in business terms.

**What you should see:** recall on the `bad` class lands around **0.25**. Out of 59 genuinely bad loans in the holdout set, the model catches about 15 and waves roughly 44 through. Meanwhile it gets 94 percent of the `good` cases right.

Sit with that. The model achieves its headline 73 percent almost entirely by agreeing with the majority class, which is the trivial baseline's strategy with extra steps. **On the actual business question, "which loans will go bad", it is close to useless.**

**Success criteria**
- [ ] Confusion matrix produced with labelled classes.
- [ ] Precision, recall, and F1 reported for the `bad` class specifically, not averaged.
- [ ] The count of missed bad loans stated explicitly.
- [ ] A recommended metric justified by the cost of each error, not by convention.

## Challenge 4: Apply the whole day

Take the taxi data from Activities 2 and 3 and build a small end to end quality pipeline that:

1. Loads the raw 30 minute data.
2. Detects and reports missing values, including any disguised ones.
3. Detects outliers using a method you can justify.
4. Repairs only what you judge should be repaired, preserving originals and flags.
5. Writes a clean Parquet file plus a short quality report of counts and decisions.

Then answer, in writing: which step in that pipeline could AutoML have done for you?

**Success criteria**
- [ ] Runs end to end from raw file to written Parquet.
- [ ] Original values and repair flags both preserved in the output.
- [ ] Every repair decision justified against the repair-or-preserve table in Activity 3.
- [ ] A written answer identifying which steps AutoML could and could not have automated.

## Challenge 5: Advise the business

Your VP has read that AutoML lets a company build models without hiring data scientists, and asks why the team still needs data engineers.

Write no more than 300 words using evidence from today. Reference the leakage result specifically, with the actual numbers.

Be fair to AutoML. It is genuinely useful, and overstating your case will not survive scrutiny. Argue what the tools do well and where the boundary sits.

**Success criteria**
- [ ] The 94 versus 73 versus 70 percent numbers used specifically, not gestured at.
- [ ] At least one genuine strength of AutoML acknowledged.
- [ ] A clear statement of where the boundary between tool and engineer sits.
- [ ] Under 300 words, written for a non-technical executive.

---
## What you did

- Established the trivial baseline before running any model, and used it to judge everything after.
- Ran a full AutoML experiment and read the leaderboard as a diagnostic.
- Obtained 94 percent accuracy, distrusted it, investigated it, and found target leakage in the `id` column.
- Showed the honest improvement was about **3 points**, not 24.
- Confirmed AutoML never questioned whether a row identifier belonged in the model.
- Toured the artifacts and learned to read a bunched leaderboard as a statement about the data.
- Learned that PyCaret is not installable on this project, and why compatibility is an architectural constraint.

## Closing the day

Four activities, one recurring theme.

In Activity 1, KNN imputation lost to the mean, and only measurement revealed it. In Activity 2, every statistical method missed the NYC Marathon, and the cause was an aggregation decision made before any detection ran. In Activity 3, Isolation Forest and Mahalanobis flagged completely different rows, and one engineered feature beat all hyperparameter tuning. In Activity 4, AutoML reported an excellent model built on a column that meant nothing.

In every case the tool did what it was asked. In every case the value came from someone deciding what to ask, checking the answer against a baseline, and understanding what the data actually meant.

That is the job. The libraries change every few years. That discipline does not.